# Stored execution evidence — RANZCR CLiP (inference template)

This public evidence copy preserves every textual output cell supplied in `mle_final_submission.zip`.
The original notebook SHA-256 is `afdeef888ac4dde6f852e36834cd760a5059d5f483e8453d26c33234ba683bf9`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# MLE-STAR RANZCR CLiP — Kaggle hidden-test inference

This is the Kaggle-side companion to the preserved Colab MLE-STAR experiment.
It does **not** train or select models. It restores the exact final checkpoints,
per-seed ensemble weights and preprocessing contract produced in Colab.

Before running, attach both inputs:

1. Competition data: `ranzcr-clip-catheter-line-classification`
2. The private Dataset created from `MLE_STAR_RANZCR_Kaggle_Assets.zip`

Select a GPU accelerator, keep Internet off, run interactively once, then use
**Save Version → Save & Run All**. The last cell creates
`/kaggle/working/submission.csv` for the code-only hidden-test submission.


In [ ]:
# Locate the hidden-test competition input and private MLE-STAR model assets.

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from torch.utils.data import DataLoader, Dataset

KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
COMPETITION_SLUG = "ranzcr-clip-catheter-line-classification"

competition_roots = [
    KAGGLE_INPUT / "competitions" / COMPETITION_SLUG,
    KAGGLE_INPUT / COMPETITION_SLUG,
]
COMPETITION_ROOT = next(
    (root for root in competition_roots if (root / "sample_submission.csv").is_file()),
    None,
)
if COMPETITION_ROOT is None:
    raise FileNotFoundError(
        "Attach the RANZCR CLiP competition data to this Kaggle Notebook."
    )

manifest_candidates = []
manifest_candidates.extend(KAGGLE_INPUT.glob("mle_star_ranzcr_manifest.json"))
manifest_candidates.extend(KAGGLE_INPUT.glob("*/mle_star_ranzcr_manifest.json"))
manifest_candidates.extend(KAGGLE_INPUT.glob("*/*/mle_star_ranzcr_manifest.json"))
manifest_candidates.extend(KAGGLE_INPUT.glob("datasets/*/*/mle_star_ranzcr_manifest.json"))

if not manifest_candidates:
    zip_candidates = [
        path
        for pattern in ("*.zip", "*/*.zip", "*/*/*.zip", "datasets/*/*/*.zip")
        for path in KAGGLE_INPUT.glob(pattern)
        if "ranzcr" in path.name.lower() and "asset" in path.name.lower()
    ]
    if not zip_candidates:
        raise FileNotFoundError(
            "Attach the private Dataset made from MLE_STAR_RANZCR_Kaggle_Assets.zip."
        )
    extracted = WORKING_DIR / "mle_star_ranzcr_assets"
    extracted.mkdir(parents=True, exist_ok=True)
    print("Extracting model assets:", zip_candidates[0])
    with zipfile.ZipFile(zip_candidates[0]) as archive:
        archive.extractall(extracted)
    manifest_candidates = list(extracted.glob("mle_star_ranzcr_manifest.json"))

if len(manifest_candidates) != 1:
    raise RuntimeError(f"Expected exactly one MLE-STAR manifest, found {manifest_candidates}")

MANIFEST_PATH = manifest_candidates[0]
ASSET_DIR = MANIFEST_PATH.parent
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest.get("competition") != COMPETITION_SLUG:
    raise ValueError("The attached model assets belong to a different competition")

SAMPLE_PATH = COMPETITION_ROOT / "sample_submission.csv"
sample = pd.read_csv(SAMPLE_PATH)
id_column = manifest["id_column"]
target_columns = list(manifest["target_columns"])
expected_columns = [id_column, *target_columns]
if sample.columns.tolist() != expected_columns:
    raise ValueError(
        f"Hidden sample schema mismatch: expected {expected_columns}, got {sample.columns.tolist()}"
    )

TEST_DIR = COMPETITION_ROOT / "test"
if not TEST_DIR.is_dir():
    raise FileNotFoundError(f"Hidden test image directory was not found: {TEST_DIR}")

test_ids = sample[id_column].astype(str).tolist()
suffix = next(
    (extension for extension in (".jpg", ".png", ".jpeg") if (TEST_DIR / f"{test_ids[0]}{extension}").is_file()),
    None,
)
if suffix is None:
    raise FileNotFoundError(f"First hidden test image was not found for {test_ids[0]}")
test_paths = [TEST_DIR / f"{value}{suffix}" for value in test_ids]
missing = [str(path) for path in test_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Hidden test images are incomplete, examples: {missing[:5]}")

for plan in manifest["plans"]:
    for item in plan["models"]:
        checkpoint_path = ASSET_DIR / item["checkpoint_file"]
        if not checkpoint_path.is_file():
            raise FileNotFoundError(checkpoint_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Select a Kaggle GPU accelerator before running inference")

print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("competition root:", COMPETITION_ROOT)
print("asset directory:", ASSET_DIR)
print("hidden test rows:", len(sample))
print("target labels:", len(target_columns))
print("seeds:", [plan["seed"] for plan in manifest["plans"]])


In [ ]:
# Restore each exact final model, reproduce the Colab preprocessing and ensemble predictions.

class RanzcrHiddenTestDataset(Dataset):
    def __init__(self, paths, image_size):
        self.paths = list(paths)
        self.image_size = int(image_size)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as opened:
            image = opened.convert("RGB").resize((self.image_size, self.image_size))
            array = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(array).permute(2, 0, 1)


dataset = RanzcrHiddenTestDataset(test_paths, manifest["image_size"])
loader = DataLoader(
    dataset,
    batch_size=int(manifest["batch_size"]),
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)


def predict_checkpoint(model_name, checkpoint_file):
    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=len(target_columns),
    )
    try:
        checkpoint = torch.load(checkpoint_file, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_file, map_location="cpu")
    if checkpoint.get("model_name") != model_name:
        raise ValueError(
            f"Checkpoint model mismatch: expected {model_name}, got {checkpoint.get('model_name')}"
        )
    load_result = model.load_state_dict(checkpoint["model"], strict=True)
    if load_result.missing_keys or load_result.unexpected_keys:
        raise RuntimeError(
            f"Strict checkpoint load failed: missing={load_result.missing_keys}, "
            f"unexpected={load_result.unexpected_keys}"
        )
    model = model.to(device).eval()
    batches = []
    with torch.inference_mode():
        for batch_index, images in enumerate(loader, start=1):
            images = images.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                probabilities = torch.sigmoid(model(images))
            batches.append(probabilities.float().cpu().numpy())
            if batch_index == 1 or batch_index % 50 == 0 or batch_index == len(loader):
                print(
                    f"[infer] model={model_name} batch={batch_index}/{len(loader)}"
                )
    prediction = np.concatenate(batches, axis=0).astype(np.float64)
    del model
    torch.cuda.empty_cache()
    return prediction


seed_predictions = []
for plan in manifest["plans"]:
    seed = int(plan["seed"])
    seed_prediction = np.zeros((len(sample), len(target_columns)), dtype=np.float64)
    total_weight = 0.0
    for item in plan["models"]:
        weight = float(item["weight"])
        model_name = item["model_name"]
        checkpoint_file = ASSET_DIR / item["checkpoint_file"]
        print(f"Loading seed={seed} model={model_name} weight={weight:.8f}")
        seed_prediction += weight * predict_checkpoint(model_name, checkpoint_file)
        total_weight += weight
    if not np.isclose(total_weight, 1.0, rtol=0, atol=1e-10):
        raise AssertionError(f"Seed {seed} model weights sum to {total_weight}")
    seed_predictions.append(seed_prediction)
    print(f"Completed seed {seed}")

raw = np.stack(seed_predictions, axis=0)
mean_prediction = raw.mean(axis=0)
if mean_prediction.shape != (len(sample), len(target_columns)):
    raise ValueError(f"Prediction shape is wrong: {mean_prediction.shape}")
if not np.isfinite(mean_prediction).all():
    raise ValueError("Predictions contain NaN or infinity")
if not ((mean_prediction >= 0.0) & (mean_prediction <= 1.0)).all():
    raise ValueError("Predictions are outside [0, 1]")

submission = sample.copy()
submission[target_columns] = mean_prediction
if submission.columns.tolist() != expected_columns:
    raise AssertionError("Submission columns changed unexpectedly")
if submission[id_column].astype(str).tolist() != test_ids:
    raise AssertionError("Submission IDs or row order changed unexpectedly")

SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
np.save(WORKING_DIR / "raw_seed_predictions.npy", raw)

print("\n=== KAGGLE SUBMISSION READY ===")
print("file:", SUBMISSION_PATH)
print("shape:", submission.shape)
print("probability range:", float(mean_prediction.min()), "to", float(mean_prediction.max()))
print("probability std:", float(mean_prediction.std()))
display(submission.head())
